In [1]:
from datasets import load_dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/crop_disease_dataset"


# loading the data
from datasets import load_from_disk

train_dataset = load_from_disk(f"{SAVE_PATH}/train")
valid_dataset = load_from_disk(f"{SAVE_PATH}/valid")
test_dataset = load_from_disk(f"{SAVE_PATH}/test")

In [ ]:
print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test :", len(test_dataset))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import timm

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224 for EfficientNet-B0
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class PlantDataset(torch.utils.data.Dataset):

    def __init__(self, hf_dataset, transform=None):

        self.ds = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):

        image = self.ds[idx]["image"].convert("RGB")
        label = self.ds[idx]["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_ds = PlantDataset(
    train_dataset,
    train_transform
)

val_ds = PlantDataset(
    valid_dataset,
    val_transform
)

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

In [ ]:
num_classes = len(
    set(train_dataset['label_name'])
)

print(num_classes)

In [ ]:
model = timm.create_model(  # EfficientNet-B0 Model
    "efficientnet_b0",
    pretrained=True
)

In [ ]:
print(model)

In [ ]:
for param in model.parameters():  # Freeze Entire Feature Extractor
    param.requires_grad = False

In [ ]:


# Unfreeze last EfficientNet blocks
for param in model.blocks[-1:].parameters():
    param.requires_grad = True

# Unfreeze final feature head
for param in model.conv_head.parameters():
    param.requires_grad = True

for param in model.bn2.parameters():
    param.requires_grad = True

In [ ]:
in_features = model.classifier.in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.3),

    nn.Linear(
        in_features,
        512
    ),

    nn.ReLU(),

    nn.Dropout(0.3),

    nn.Linear(
        512,
        num_classes
    )
)

In [ ]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

ratio = trainable_params / total_params

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Ratio                : {ratio:.4f}")
print(f"Percentage           : {ratio*100:.2f}%")

In [ ]:
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
from tqdm.auto import tqdm

def train_one_epoch():

    model.train()

    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc="Training",
        leave=False
    )

    for images, labels in pbar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()


        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()


        total_loss += loss.item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / len(train_loader)

In [ ]:
from tqdm.auto import tqdm

def validate():

    model.eval()

    correct = 0
    total = 0

    pbar = tqdm(
        val_loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for images, labels in pbar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

            current_acc = correct / total

            pbar.set_postfix(
                acc=f"{current_acc:.4f}"
            )

    return correct / total

In [ ]:
import os

save_dir = "/content/drive/MyDrive/CropDiseaseModels"

In [ ]:
epochs = 100
patience = 5
min_delta = 0.0001

train_loss_history = []
val_acc_history = []

best_val_acc = -float("inf")
best_epoch = 0

counter = 0


for epoch in range(epochs):

    loss = train_one_epoch()

    acc = validate()

    train_loss_history.append(loss)
    val_acc_history.append(acc)

    print(
        f"Epoch {epoch+1}: "
        f"Loss={loss:.4f} "
        f"Val Acc={acc:.4f}"
    )


    if acc > (best_val_acc + min_delta):
      best_val_acc = acc
      best_epoch = epoch + 1

      counter = 0

      torch.save(
            model.state_dict(),
            f"{save_dir}/cnn_model_weights_v2.pth"
                  )

      torch.save(
          model,
          f"{save_dir}/cnn_full_model_v2.pth"
      )
      print("Model saved successfully.")

    else:

        counter += 1

        print(
            f"No improvement for {counter} epoch(s)."
        )


    # Early stopping

    if counter >= patience:

        print("\nEarly stopping triggered.")
        break

print(f"\nBest validation accuracy : {best_val_acc:.6f}")
print(f"Best epoch           : {best_epoch}")